# 支付表清洗：第一步，粒度、组合主键与支付记录结构

一行代表一条订单支付记录。一笔订单可能有多条记录，先检查 `order_id + payment_sequential` 能否共同唯一识别记录。

| 字段 | 含义 |
| --- | --- |
| order_id | 订单 ID，用于关联订单表 |
| payment_sequential | 订单内支付记录序号 |
| payment_type | 支付方式，例如信用卡、借记卡、银行付款单、代金券 |
| payment_installments | 该条记录的分期期数 |
| payment_value | 该条支付记录的总金额，单位 BRL |

**支付记录条数、支付记录序号和分期期数是三个概念。**例如，一条信用卡记录的分期期数为 4，金额为 665.41 BRL，则这条记录的总金额仍为 665.41 BRL，不能再乘以 4。

本次保留原始字符串，只观察和检查。后续再转换金额、检查未知类别和期数、汇总到订单粒度并对账。

In [6]:
from pathlib import Path
import pandas as pd

project_dir = Path(r'D:\数据分析\olist-operations-analytics')
data_dir = project_dir.parent / 'archive'

payments_raw = pd.read_csv(
    data_dir / 'olist_order_payments_dataset.csv',
    dtype=str, encoding='utf-8-sig',
    keep_default_na=False, na_values=['']
)
print('行数和列数：', payments_raw.shape)
payments_raw.head(3)

行数和列数： (103886, 5)


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


In [7]:
payments_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   order_id              103886 non-null  object
 1   payment_sequential    103886 non-null  object
 2   payment_type          103886 non-null  object
 3   payment_installments  103886 non-null  object
 4   payment_value         103886 non-null  object
dtypes: object(5)
memory usage: 4.0+ MB


In [3]:
payment_key_cols = ['order_id', 'payment_sequential']

payment_key_check = pd.Series({
    '支付记录行数': len(payments_raw),
    '涉及订单数': payments_raw['order_id'].nunique(),
    '订单ID缺失行数': payments_raw['order_id'].isna().sum(),
    '支付序号缺失行数': payments_raw['payment_sequential'].isna().sum(),
    '组合主键重复数_首条之后': payments_raw.duplicated(subset=payment_key_cols).sum(),
    '整行重复数_首条之后': payments_raw.duplicated().sum()
})

payment_key_check

支付记录行数          103886
涉及订单数            99440
订单ID缺失行数             0
支付序号缺失行数             0
组合主键重复数_首条之后         0
整行重复数_首条之后           0
dtype: int64

## 一笔订单有几条支付记录

先用 `value_counts()` 计算每个订单对应的记录行数，再判断哪些订单超过一条。此处统计记录数量，不使用 `payment_installments`。

In [4]:
payment_counts = payments_raw['order_id'].value_counts()

print('只有一条支付记录的订单数：', payment_counts.eq(1).sum())
print('有多条支付记录的订单数：', payment_counts.gt(1).sum())
print('单笔订单最多支付记录数：', payment_counts.max())

只有一条支付记录的订单数： 96479
有多条支付记录的订单数： 2961
单笔订单最多支付记录数： 29


In [5]:
# 找一笔恰好有两条支付记录的真实订单，观察五个字段。
example_order = payment_counts[payment_counts.eq(2)].index[0]
payments_raw.loc[payments_raw['order_id'].eq(example_order)]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
48202,95fbd229320a05e8835ea19d93ba9f1e,1,credit_card,1,44.47
76987,95fbd229320a05e8835ea19d93ba9f1e,2,voucher,1,100.00


## 我的理解

1. 支付记录有多少行，涉及多少个订单？组合主键是否非空且唯一？

我的回答：支付记录有103886条，涉及99440个订单，组合主键非空且唯一。

2. `payment_sequential` 与 `payment_installments` 有什么区别？为什么不能用支付记录条数当作分期期数？

我的回答：payment_sequential是订单内支付记录序号，就是同一个order_id不同的payment_sequential，payment_installments是该条记录的分期期数。
一笔订单可以有多条支付记录，而每条支付记录又可能有自己的分期期数，所以支付记录条数不能代表分期期数。

3. 某条支付记录的分期期数为 4、payment_value 为 665.41，汇总订单支付额时，这条记录应计入多少金额？

我的回答：计入665.41

核对基线：103,886 条记录、99,440 个订单；组合键无缺失、无重复；2,961 个订单有多条记录，单笔最多 29 条。

支付表覆盖的订单数比订单主表少 1，留到订单覆盖检查时定位，先不填补。字段无空值不等于所有订单都有支付记录。

下一步：payment_value 的金额检查与整数分转换、支付方式和期数取值检查，再按订单汇总并与商品金额加运费核对。

In [8]:
payments_clean = payments_raw.copy()

# 临时转换，用于检查；精确金额仍从原始字符串转换
payment_value_num = pd.to_numeric(
    payments_raw['payment_value'],
    errors='coerce'
)

payment_money_check = pd.Series({
    '原始缺失数': payments_raw['payment_value'].isna().sum(),
    '非空但解析失败数': (
        payments_raw['payment_value'].notna()
        & payment_value_num.isna()
    ).sum(),
    '负数记录数': payment_value_num.lt(0).sum(),
    '零值记录数': payment_value_num.eq(0).sum(),
    '最小值': payment_value_num.min(),
    '最大值': payment_value_num.max()
})

payment_money_check

原始缺失数           0.00
非空但解析失败数        0.00
负数记录数           0.00
零值记录数           9.00
最小值             0.00
最大值         13664.08
dtype: float64

In [9]:
import sys

src_dir = str(project_dir / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from items_cleaning import to_cents

payments_clean['payment_value_cents'] = (
    payments_raw['payment_value']
    .map(to_cents)
    .astype('int64')
)

payments_clean['flag_zero_payment'] = (
    payments_clean['payment_value_cents'].eq(0)
)

print('金额类型：', payments_clean['payment_value_cents'].dtype)
print('零金额支付记录数：', payments_clean['flag_zero_payment'].sum())

payments_clean['payment_value_cents'].agg(['min', 'max'])

金额类型： int64
零金额支付记录数： 9


min          0
max    1366408
Name: payment_value_cents, dtype: int64

In [10]:
payments_clean.loc[
    payments_clean['flag_zero_payment'],
    [
        'order_id',
        'payment_sequential',
        'payment_type',
        'payment_installments',
        'payment_value',
        'payment_value_cents'
    ]
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value,payment_value_cents
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00,0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00,0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00,0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00,0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00,0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00,0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00,0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00,0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.00,0


In [11]:
payments_clean['payment_type'].value_counts(dropna=False)

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [12]:
payments_clean['flag_payment_type_not_defined'] = (
    payments_clean['payment_type'].eq('not_defined')
)

In [13]:
integer_cols = ['payment_sequential', 'payment_installments']

# 先生成检查用的数值表
payment_numeric = payments_raw[integer_cols].apply(
    pd.to_numeric,
    errors='coerce'
)

payment_integer_check = pd.DataFrame({
    '原始缺失数': payments_raw[integer_cols].isna().sum(),
    '非空但解析失败数': (
        payments_raw[integer_cols].notna()
        & payment_numeric.isna()
    ).sum(),
    '非整数记录数': (
        payment_numeric.notna()
        & payment_numeric.mod(1).ne(0)
    ).sum(),
    '负数记录数': payment_numeric.lt(0).sum(),
    '零值记录数': payment_numeric.eq(0).sum(),
    '最小值': payment_numeric.min(),
    '最大值': payment_numeric.max()
})

payment_integer_check

,原始缺失数,非空但解析失败数,非整数记录数,负数记录数,零值记录数,最小值,最大值
payment_sequential,0,0,0,0,0,1,29
payment_installments,0,0,0,0,2,0,24


In [14]:
payments_clean['flag_zero_installments'] = (
    payment_numeric['payment_installments'].eq(0)
)

payments_clean.loc[
    payments_clean['flag_zero_installments'],
    [
        'order_id',
        'payment_sequential',
        'payment_type',
        'payment_installments',
        'payment_value',
        'payment_value_cents'
    ]
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value,payment_value_cents
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69,5869
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94,12994


In [15]:
integer_cols = ['payment_sequential', 'payment_installments']

for col in integer_cols:
    payments_clean[col] = payment_numeric[col].astype('Int64')

# 类型转换后，再次确认组合主键唯一
if payments_clean.duplicated(
    ['order_id', 'payment_sequential']
).any():
    raise ValueError('类型转换后组合主键重复，请核查')

payments_clean[integer_cols].dtypes

payment_sequential      Int64
payment_installments    Int64
dtype: object

In [20]:
order_payment_summary = (
    payments_clean
    .groupby('order_id', as_index=False)
    .agg(
        payment_record_count=('payment_sequential', 'size'),
        payment_amount_cents=('payment_value_cents', 'sum'),
        has_zero_payment_record=('flag_zero_payment', 'any'),
        has_undefined_payment_type=(
            'flag_payment_type_not_defined', 'any'
        ),
        has_zero_installments=('flag_zero_installments', 'any')
    )
)

print('汇总后的订单数：', len(order_payment_summary))
print('订单ID重复数：', order_payment_summary['order_id'].duplicated().sum())

order_payment_summary.head()

汇总后的订单数： 99440
订单ID重复数： 0


,order_id,payment_record_count,payment_amount_cents,has_zero_payment_record,has_undefined_payment_type,has_zero_installments
0,00010242fe8c5a6d1ba2dd792cb16214,1,7219,False,False,False
1,00018f77f2f0320c557190d7a144bdd3,1,25983,False,False,False
2,000229ec398224ef6ca0657da4fc703e,1,21687,False,False,False
3,00024acbcdf0a6daa1e931b038114c75,1,2578,False,False,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,21804,False,False,False


In [21]:
payment_reconcile = pd.DataFrame({
    '汇总前': [
        len(payments_clean),
        payments_clean['payment_value_cents'].sum()
    ],
    '汇总后': [
        order_payment_summary['payment_record_count'].sum(),
        order_payment_summary['payment_amount_cents'].sum()
    ]
}, index=['支付记录数', '支付金额（分）'])

payment_reconcile['差额'] = (
    payment_reconcile['汇总后']
    - payment_reconcile['汇总前']
)

payment_reconcile

,汇总前,汇总后,差额
支付记录数,103886,103886,0
支付金额（分）,1600887212,1600887212,0


In [22]:
orders_base = pd.read_csv(
    data_dir / 'olist_orders_dataset.csv',
    dtype=str,
    encoding='utf-8-sig',
    keep_default_na=False,
    na_values=['']
)

orphan_payment_orders = order_payment_summary.loc[
    ~order_payment_summary['order_id'].isin(orders_base['order_id'])
]

print('支付汇总中找不到主表的订单数：', len(orphan_payment_orders))

if not orphan_payment_orders.empty:
    raise ValueError('存在无法匹配订单主表的支付订单，请先核查')

支付汇总中找不到主表的订单数： 0


In [23]:
orders_with_payments = orders_base.merge(
    order_payment_summary,
    on='order_id',
    how='left',
    validate='one_to_one',
    indicator='payments_match'
)

# 使用可保留缺失值的整数类型
for col in ['payment_record_count', 'payment_amount_cents']:
    orders_with_payments[col] = (
        orders_with_payments[col].astype('Int64')
    )

orders_with_payments['flag_missing_payments'] = (
    orders_with_payments['payments_match'].eq('left_only')
)

print('关联后订单行数：', len(orders_with_payments))
print(orders_with_payments['payments_match'].value_counts())

关联后订单行数： 99441
payments_match
both          99440
left_only         1
right_only        0
Name: count, dtype: int64


In [24]:
orders_with_payments.loc[
    orders_with_payments['flag_missing_payments'],
    [
        'order_id',
        'order_status',
        'order_purchase_timestamp',
        'order_delivered_customer_date',
        'payment_record_count',
        'payment_amount_cents'
    ]
]

,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,payment_record_count,payment_amount_cents
30710,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38,2016-11-09 07:47:38,<NA>,<NA>


In [25]:
summary_path = (
    project_dir / 'data' / 'processed' / 'items_v1'
    / 'order_item_summary_v1.csv'
)

order_item_summary = pd.read_csv(
    summary_path,
    usecols=[
        'order_id',
        'goods_amount_cents',
        'freight_amount_cents'
    ],
    dtype={
        'order_id': str,
        'goods_amount_cents': 'Int64',
        'freight_amount_cents': 'Int64'
    },
    encoding='utf-8-sig'
)

orders_amount_check = orders_with_payments.merge(
    order_item_summary,
    on='order_id',
    how='left',
    validate='one_to_one',
    indicator='items_match'
)

print('关联后订单行数：', len(orders_amount_check))

关联后订单行数： 99441


In [26]:
amount_cols = [
    'goods_amount_cents',
    'freight_amount_cents',
    'payment_amount_cents'
]

for col in amount_cols:
    orders_amount_check[col] = (
        orders_amount_check[col].astype('Int64')
    )

# 三个金额字段全部非空，才具备核对条件
orders_amount_check['can_reconcile_amount'] = (
    orders_amount_check[amount_cols]
    .notna()
    .all(axis=1)
)

orders_amount_check['items_total_cents'] = (
    orders_amount_check['goods_amount_cents']
    + orders_amount_check['freight_amount_cents']
)

orders_amount_check['payment_difference_cents'] = (
    orders_amount_check['payment_amount_cents']
    - orders_amount_check['items_total_cents']
)

In [27]:
comparable_orders = orders_amount_check.loc[
    orders_amount_check['can_reconcile_amount']
].copy()

difference = comparable_orders['payment_difference_cents']

amount_check_summary = pd.Series({
    '总订单数': len(orders_amount_check),
    '可核对订单数': len(comparable_orders),
    '无法核对订单数': (
        ~orders_amount_check['can_reconcile_amount']
    ).sum(),
    '金额完全一致订单数': difference.eq(0).sum(),
    '存在金额差额订单数': difference.ne(0).sum()
})

amount_check_summary

总订单数         99441
可核对订单数       98665
无法核对订单数        776
金额完全一致订单数    98089
存在金额差额订单数      576
dtype: int64

In [28]:
difference_orders = comparable_orders.loc[
    comparable_orders['payment_difference_cents'].ne(0)
].copy()

difference_orders['abs_difference_cents'] = (
    difference_orders['payment_difference_cents'].abs()
)

diff = difference_orders['payment_difference_cents']
abs_diff = difference_orders['abs_difference_cents']

difference_distribution = pd.Series({
    '存在差额订单数': len(difference_orders),
    '支付额较大订单数': diff.gt(0).sum(),
    '支付额较小订单数': diff.lt(0).sum(),
    '绝对差额为1分': abs_diff.eq(1).sum(),
    '绝对差额为2至99分': abs_diff.between(2, 99).sum(),
    '绝对差额为100分及以上': abs_diff.ge(100).sum()
})

difference_distribution

存在差额订单数         576
支付额较大订单数        367
支付额较小订单数        209
绝对差额为1分         273
绝对差额为2至99分       53
绝对差额为100分及以上    250
dtype: int64

In [29]:
top_difference_orders = difference_orders.nlargest(
    5,
    'abs_difference_cents'
).copy()

top_difference_orders[
    [
        'order_id',
        'order_status',
        'goods_amount_cents',
        'freight_amount_cents',
        'items_total_cents',
        'payment_amount_cents',
        'payment_difference_cents'
    ]
]

,order_id,order_status,goods_amount_cents,freight_amount_cents,items_total_cents,payment_amount_cents,payment_difference_cents
11791,ce6d150fb29ada17d2082f4847107665,delivered,129900,10466,140366,158647,18281
48686,6e5fe7366a2e1bfbf3257dba0af1267f,delivered,17919,10872,28791,40692,11901
70865,70b742795bc441e94a44a084b6d9ce7a,delivered,26999,19694,46693,57882,11189
33150,996c7e73600ad3723e8627ab7bef81e4,delivered,55990,2800,58790,66443,7653
52985,70b7e94ea46d3e8b5bc12a50186edaf0,delivered,16788,4527,21315,27484,6169


In [30]:
from IPython.display import display

check_order_id = top_difference_orders.iloc[0]['order_id']

items_raw = pd.read_csv(
    data_dir / 'olist_order_items_dataset.csv',
    dtype=str,
    encoding='utf-8-sig',
    keep_default_na=False,
    na_values=['']
)

example_items = items_raw.loc[
    items_raw['order_id'].eq(check_order_id)
].copy()

example_payments = payments_raw.loc[
    payments_raw['order_id'].eq(check_order_id)
].copy()

display(example_items[
    ['order_id', 'order_item_id', 'price', 'freight_value']
])

display(example_payments[
    [
        'order_id',
        'payment_sequential',
        'payment_type',
        'payment_installments',
        'payment_value'
    ]
])

,order_id,order_item_id,price,freight_value
90875,ce6d150fb29ada17d2082f4847107665,1,1299.00,104.66


,order_id,payment_sequential,payment_type,payment_installments,payment_value
359,ce6d150fb29ada17d2082f4847107665,1,credit_card,10,1586.47


In [31]:
raw_goods_cents = int(
    example_items['price'].map(to_cents).sum()
)

raw_freight_cents = int(
    example_items['freight_value'].map(to_cents).sum()
)

raw_payment_cents = int(
    example_payments['payment_value'].map(to_cents).sum()
)

raw_difference_cents = (
    raw_payment_cents - raw_goods_cents - raw_freight_cents
)

summary_row = top_difference_orders.iloc[0]

case_recheck = pd.DataFrame({
    '原始明细重算': [
        raw_goods_cents,
        raw_freight_cents,
        raw_payment_cents,
        raw_difference_cents
    ],
    '订单级结果': [
        int(summary_row['goods_amount_cents']),
        int(summary_row['freight_amount_cents']),
        int(summary_row['payment_amount_cents']),
        int(summary_row['payment_difference_cents'])
    ]
}, index=['商品金额（分）', '运费（分）', '支付金额（分）', '金额差额（分）'])

case_recheck['差值'] = (
    case_recheck['原始明细重算']
    - case_recheck['订单级结果']
)

case_recheck

,原始明细重算,订单级结果,差值
商品金额（分）,129900,129900,0
运费（分）,10466,10466,0
支付金额（分）,158647,158647,0
金额差额（分）,18281,18281,0
